In [1]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [3]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        # Zjistíme embedding dimenzi na prvním SMILES
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("CK1/data/ck1_train_1")

X2_all = load_datasets("CK1/data/ck1_val_1")

X3_all = load_datasets("CK1/data/ck1_test_1")

#transformer = ChemBERTaTransformer()
#X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
#X_val_emb = transformer.transform(X2_all.df["Drug"])
#X_test_emb = transformer.transform(X3_all.df["Drug"])

In [5]:
display(X2_all.df)

,QSPRID,Y,Drug,Y_original
QSPRID,,,,
A2ARDataset_000,A2ARDataset_000,False,COc1cc(C)c(NC(=O)CN(C)C)cc1Nc1nc(Nc2cccc(F)c2C...,False
A2ARDataset_001,A2ARDataset_001,False,CC(C)(C)c1cc(NC(=O)C(=O)c2cccc3ccccc23)n(-c2cc...,False
A2ARDataset_002,A2ARDataset_002,False,Nc1nnc(-c2cc3c(Oc4ccc(Cl)cc4)cncc3s2)o1,False
A2ARDataset_003,A2ARDataset_003,False,OCc1ccc(-c2nc(-c3ccccn3)c(-c3ccc4c(c3)OCO4)[nH...,False
A2ARDataset_004,A2ARDataset_004,False,CS(=O)(=O)CCNCc1ccoc1-c1ccc2ncnc(Nc3ccc(OCc4cc...,False
...,...,...,...,...
A2ARDataset_150,A2ARDataset_150,False,COc1cc2nc3nc(-c4cccs4)c(-c4cccs4)nc3nc2cc1OC,False
A2ARDataset_151,A2ARDataset_151,False,NC1(C(=O)NCc2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1,False
A2ARDataset_152,A2ARDataset_152,False,CCc1cccc(NC(=O)Nc2ccc(Oc3ccc4nc(NC(=O)OC)[nH]c...,False


In [6]:
from MolEval import MolEmb 
model_name = 'Morgan'  # Replace with the model you want to use

X1_all.df["SMILES"] = X1_all.df["Drug"]
extractor = MolEmb.EmbeddingExtractor(model_name=model_name, df=X1_all.df)
emb, X1_all.df = extractor.get_embeddings()
print(emb)


No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


     0     1     2     3     4     5     6     7     8     9     ...  1014  \
0       0     0     0     0     0     0     0     0     0     0  ...     0   
1       0     0     0     0     0     0     0     0     0     0  ...     0   
2       0     0     0     0     0     0     0     0     0     0  ...     0   
3       0     0     0     0     0     0     0     0     0     0  ...     0   
4       0     0     0     0     0     0     0     0     0     0  ...     0   
..    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   ...   
483     0     0     0     0     0     0     0     0     0     0  ...     0   
484     0     0     0     0     0     0     0     0     0     0  ...     0   
485     0     0     1     0     0     0     0     0     0     0  ...     0   
486     0     0     0     0     1     0     0     0     0     0  ...     0   
487     0     0     0     0     0     0     0     0     0     0  ...     0   

     1015  1016  1017  1018  1019  1020  1021  1022  1023  
0  

In [7]:
display(X1_all.X.iloc[:, :1024])

,MorganFP_MorganFP_0,MorganFP_MorganFP_1,MorganFP_MorganFP_2,MorganFP_MorganFP_3,MorganFP_MorganFP_4,MorganFP_MorganFP_5,MorganFP_MorganFP_6,MorganFP_MorganFP_7,MorganFP_MorganFP_8,MorganFP_MorganFP_9,...,MorganFP_MorganFP_1014,MorganFP_MorganFP_1015,MorganFP_MorganFP_1016,MorganFP_MorganFP_1017,MorganFP_MorganFP_1018,MorganFP_MorganFP_1019,MorganFP_MorganFP_1020,MorganFP_MorganFP_1021,MorganFP_MorganFP_1022,MorganFP_MorganFP_1023
QSPRID,,,,,,,,,,,,,,,,,,,,,
A2ARDataset_000,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
A2ARDataset_001,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
A2ARDataset_002,False,True,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
A2ARDataset_003,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
A2ARDataset_004,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A2ARDataset_483,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
A2ARDataset_484,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
A2ARDataset_485,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False


In [8]:
X2_all.df["SMILES"] = X2_all.df["Drug"]
extractor2 = MolEmb.EmbeddingExtractor(model_name=model_name, df=X2_all.df)
emb2, X2_all.df = extractor2.get_embeddings()
print(emb2)


     0     1     2     3     4     5     6     7     8     9     ...  1014  \
0       0     0     1     0     0     0     0     0     1     0  ...     0   
1       0     0     0     0     0     0     0     0     0     0  ...     0   
2       0     0     0     0     0     0     0     0     0     0  ...     0   
3       0     0     0     0     0     0     0     0     0     0  ...     0   
4       0     0     0     0     0     0     0     0     0     0  ...     0   
..    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   ...   
150     0     0     0     0     0     0     0     0     0     0  ...     0   
151     0     0     1     0     0     0     0     0     0     0  ...     0   
152     0     0     0     0     0     0     0     0     0     0  ...     0   
153     0     1     0     0     0     1     0     0     0     0  ...     0   
154     0     0     0     0     0     0     0     0     0     0  ...     0   

     1015  1016  1017  1018  1019  1020  1021  1022  1023  
0  

In [9]:
X3_all.df["SMILES"] = X3_all.df["Drug"]
extractor3 = MolEmb.EmbeddingExtractor(model_name=model_name, df=X3_all.df)
emb3, X3_all.df = extractor3.get_embeddings()
print(emb3)


     0     1     2     3     4     5     6     7     8     9     ...  1014  \
0       0     0     0     0     0     0     0     0     0     0  ...     0   
1       0     0     0     0     0     0     0     0     0     0  ...     0   
2       0     0     0     0     0     0     0     0     0     0  ...     0   
3       0     0     0     0     0     0     0     0     0     0  ...     0   
4       0     0     0     0     0     0     0     0     0     0  ...     0   
..    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   ...   
159     0     0     1     0     0     0     0     0     0     0  ...     1   
160     0     0     0     0     0     0     0     0     0     0  ...     0   
161     0     0     0     0     0     0     0     0     0     0  ...     1   
162     0     0     0     0     0     0     0     0     0     0  ...     1   
163     0     0     0     0     0     0     0     0     0     0  ...     0   

     1015  1016  1017  1018  1019  1020  1021  1022  1023  
0  

In [10]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
emb = emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, emb], axis = 1)

In [11]:
display(X1_all.X)

,MorganFP_MorganFP_0,MorganFP_MorganFP_1,MorganFP_MorganFP_2,MorganFP_MorganFP_3,MorganFP_MorganFP_4,MorganFP_MorganFP_5,MorganFP_MorganFP_6,MorganFP_MorganFP_7,MorganFP_MorganFP_8,MorganFP_MorganFP_9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,False,False,False,False,False,False,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
1,False,False,False,False,False,False,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
2,False,True,False,False,False,False,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
3,False,False,False,False,False,False,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
4,False,False,False,True,False,False,False,False,False,False,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,False,False,False,False,False,False,False,False,False,False,...,0,0,0,0,0,1,0,0,0,0
484,False,False,False,False,False,False,False,False,False,False,...,0,0,0,0,0,0,0,0,0,0
485,False,False,False,False,False,False,False,False,True,False,...,0,0,0,0,0,1,0,0,0,0
486,False,False,False,False,False,False,False,False,False,False,...,0,0,1,0,0,1,0,0,0,0


In [12]:
X2_all.X = X2_all.X.reset_index(drop=True)
emb2 = emb2.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, emb2], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
emb3 = emb3.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, emb3], axis = 1)

In [13]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [14]:
X1.columns = X1.columns.astype(str)
X2.columns = X2.columns.astype(str)
X3.columns = X3.columns.astype(str)

imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [15]:
pd.DataFrame(X1).columns[pd.DataFrame(X1).isna().any()].tolist()



[]

In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,2248,2249,2250,2251,2252,2253,2254,2255,2256,2257
0,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,-0.232370,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
1,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,-0.232370,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
2,-0.090909,2.722828,-0.090909,-0.246718,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,-0.232370,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
3,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,-0.232370,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
4,-0.090909,-0.367265,-0.090909,4.053217,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,-0.232370,5.109903,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,4.303487,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
591,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,4.303487,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
592,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137073,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,-0.232370,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415
593,-0.090909,-0.367265,-0.090909,-0.246718,0.0,6.992298,0.0,-0.090909,-0.251358,-0.111571,...,-0.246718,-0.078648,-0.232370,-0.195698,-0.17186,-0.564729,-0.129099,-0.06415,-0.129099,-0.06415


In [17]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [18]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, matthews_corrcoef
import pandas as pd

def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    val_mcc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        val_mcc_t.append(matthews_corrcoef(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
        print(matthews_corrcoef(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    my_df["MCC"] = val_mcc_t
    return my_df

In [5]:
import optuna
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
import torch
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import confusion_matrix
def objective(trial, X_train, y_train, X_test, y_test):
    dropout_frac = trial.suggest_categorical("dropout_frac", [0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.8, 0.9])
    patience = trial.suggest_categorical("patience", [10, 40, 75])
    tol = trial.suggest_categorical("tol", [1e-5, 1e-4, 1e-3, 1e-2, 0])
    weight_decay = trial.suggest_categorical("weight_decay", [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 0])
    n_epochs = trial.suggest_categorical("n_epochs", [200, 300, 500, 1000])
    batch_size = trial.suggest_categorical("batch_size", [1024, 512, 256, 128, 64])
    optimizer = trial.suggest_categorical("optimizer", ["optim.AdamW", "optim.RMSprop"])
    lr = trial.suggest_categorical("lr", [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
    neuron_layers_dict = {
    '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 1024, 256, 64, 8]': [4096, 1024, 256, 64, 8],
    '[4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8]': [4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096]': [4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096],
    '[200]': [200],
    '[2000]': [2000],
    '[2000, 1000]': [2000, 1000],
    '[2000, 1000, 500]': [2000, 1000, 500],
    '[1000, 50]': [1000, 50],
    '[4000, 2000]': [4000, 2000],
    '[4000, 2000, 1000, 500]': [4000, 2000, 1000, 500],
    '[4000, 2000, 2000, 500]': [4000, 2000, 2000, 500]
    }
    neuron_layers_size = trial.suggest_categorical("neuron_layers_size", list(neuron_layers_dict.keys()))
    opt = {"optim.AdamW": optim.AdamW,
           "optim.RMSprop": optim.RMSprop}
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    
    # Model
    model = STFullyConnected(
        n_dim=X_train.shape[1],
        n_class=1,
        gpus=[],
        device=device,
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=dropout_frac,
        patience=patience,
        tol=tol,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=weight_decay,
        n_epochs=n_epochs,
        neuron_layers= neuron_layers_dict[neuron_layers_size],  # Použití neuron_layers_size
        batch_size=batch_size,
        optimizer=opt[optimizer],
        lr=lr,
        random_seed=69
    )
    # Trénink a predikce
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_bin = preds > 0.5

    # Metiky
    f1 = f1_score(y_test, preds_bin)
    acc = accuracy_score(y_test, preds_bin)
    mcc = matthews_corrcoef(y_test, preds_bin)
    
    # Můžeš logovat i do trialu
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("acc", acc)
    display(confusion_matrix(y_test, preds_bin))
    return mcc  # maximalizujeme MCC


In [6]:
study_3 = optuna.create_study(
    study_name="CK1_new_roberta",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.NSGAIISampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=200
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

[I 2025-05-05 00:23:14,692] Using an existing study with name 'CK1_new_roberta' instead of creating a new one.
[W 2025-05-05 00:23:14,767] Trial 1 failed with parameters: {} because of the following error: NameError("name 'X1' is not defined").
Traceback (most recent call last):
  File "/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_32147/1622199706.py", line 11, in <lambda>
    lambda trial: objective(trial, X1, y1, X2, y2),
                                   ^^
NameError: name 'X1' is not defined
[W 2025-05-05 00:23:14,768] Trial 1 failed with value None.


NameError: name 'X1' is not defined

In [3]:
study_3 = optuna.create_study(
    study_name="CK1_new_roberta",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.NSGAIISampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=1000
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

[I 2025-05-05 00:22:51,522] A new study created in RDB with name: CK1_new_roberta
[W 2025-05-05 00:22:51,597] Trial 0 failed with parameters: {} because of the following error: NameError("name 'X1' is not defined").
Traceback (most recent call last):
  File "/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_32147/239421387.py", line 11, in <lambda>
    lambda trial: objective(trial, X1, y1, X2, y2),
                                   ^^
NameError: name 'X1' is not defined
[W 2025-05-05 00:22:51,598] Trial 0 failed with value None.


NameError: name 'X1' is not defined

In [7]:
# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

ValueError: Record does not exist.